# Policy Gradient Methods

## Learning Objectives
1. Implement REINFORCE on a K-armed bandit and compare to epsilon-greedy
2. Build REINFORCE with value baseline on CartPole simulation with entropy monitoring
3. Extend to continuous action spaces using a Gaussian policy
4. Quantify variance reduction from baseline and reward normalization

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch: {torch.__version__}')

## Level 1: REINFORCE on K-Armed Bandit

K=5 arms with fixed reward means. REINFORCE learns a softmax policy via log-gradient.  
Compare to epsilon-greedy — both solve bandit but REINFORCE is differentiable and extends to sequences.

In [ ]:
class KArmedBandit:
    """Stationary K-armed bandit with Gaussian reward distributions."""

    def __init__(self, k: int = 5, seed: int = 42):
        rng = np.random.RandomState(seed)
        self.k = k
        # True mean rewards: arm 3 is best with mean 2.0
        self.true_means = rng.randn(k) + np.array([0.0, 0.5, 1.0, 2.0, 0.3])
        self.optimal_arm = int(np.argmax(self.true_means))

    def pull(self, arm: int) -> float:
        """Pull arm, receive noisy reward."""
        return float(np.random.randn() + self.true_means[arm])


def reinforce_bandit(
    bandit: KArmedBandit,
    n_steps: int = 1000,
    lr: float = 0.1,
    use_baseline: bool = False,
) -> tuple:
    """REINFORCE for bandit: gradient = reward * grad_log_pi. Returns (rewards, policy_probs)."""
    # Policy: softmax over learned preferences H[a]
    H = np.zeros(bandit.k)  # Action preferences (logits)
    baseline = 0.0          # Running mean of rewards
    rewards = []
    optimal_pct = []

    for step in range(n_steps):
        # Softmax policy
        exp_H = np.exp(H - H.max())  # Numerically stable softmax
        pi = exp_H / exp_H.sum()

        # Sample action from policy
        arm = np.random.choice(bandit.k, p=pi)
        reward = bandit.pull(arm)
        rewards.append(reward)
        optimal_pct.append(1 if arm == bandit.optimal_arm else 0)

        # Update baseline
        if use_baseline:
            baseline += 0.01 * (reward - baseline)  # Exponential moving average

        # REINFORCE gradient: H[a] += lr * (R - baseline) * (1{a==arm} - pi[a])
        advantage = reward - baseline
        for a in range(bandit.k):
            # Gradient of log pi(arm|H) w.r.t. H[a]
            grad_log_pi = (1.0 if a == arm else 0.0) - pi[a]
            H[a] += lr * advantage * grad_log_pi

    return rewards, optimal_pct, pi


def epsilon_greedy_bandit(bandit: KArmedBandit, n_steps: int = 1000, eps: float = 0.1) -> tuple:
    """Epsilon-greedy bandit baseline."""
    Q = np.zeros(bandit.k)
    counts = np.zeros(bandit.k)
    rewards = []
    optimal_pct = []
    for step in range(n_steps):
        arm = np.random.randint(bandit.k) if np.random.rand() < eps else np.argmax(Q)
        r = bandit.pull(arm)
        counts[arm] += 1
        Q[arm] += (r - Q[arm]) / counts[arm]  # Running average
        rewards.append(r)
        optimal_pct.append(1 if arm == bandit.optimal_arm else 0)
    return rewards, optimal_pct


bandit = KArmedBandit(k=5)
print(f'Bandit arm means: {bandit.true_means.round(2)}')
print(f'Optimal arm: {bandit.optimal_arm} (mean={bandit.true_means[bandit.optimal_arm]:.2f})')

np.random.seed(42)
reinf_rewards, reinf_opt, final_pi = reinforce_bandit(bandit, n_steps=1000)
np.random.seed(42)
eg_rewards, eg_opt = epsilon_greedy_bandit(bandit, n_steps=1000)

print(f'\nREINFORCE final policy: {final_pi.round(3)}')
print(f'REINFORCE optimal arm % (last 200): {np.mean(reinf_opt[-200:])*100:.1f}%')
print(f'Epsilon-greedy optimal arm % (last 200): {np.mean(eg_opt[-200:])*100:.1f}%')

## Level 2: REINFORCE with Baseline on CartPole Simulation

Full REINFORCE with a learned value baseline V(s).  
Monitor policy entropy to detect premature collapse.

In [ ]:
def cartpole_step(state: np.ndarray, action: int, dt: float = 0.02):
    """CartPole physics. State: [x, x_dot, theta, theta_dot]. action: 0=left, 1=right."""
    x, x_dot, theta, theta_dot = state
    force = 10.0 if action == 1 else -10.0
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    temp = (force + 0.05 * theta_dot**2 * sin_t) / 1.1
    theta_acc = (9.8 * sin_t - cos_t * temp) / (0.5 * (4/3 - 0.1 * cos_t**2 / 1.1))
    x_acc = temp - 0.05 * theta_acc * cos_t / 1.1
    x += dt * x_dot; x_dot += dt * x_acc
    theta += dt * theta_dot; theta_dot += dt * theta_acc
    done = abs(x) > 2.4 or abs(theta) > 0.2
    return np.array([x, x_dot, theta, theta_dot]), 1.0 if not done else 0.0, done


def cartpole_reset():
    return np.random.uniform(-0.05, 0.05, 4)


class PolicyNetwork(nn.Module):
    """Policy network: outputs log-probabilities over discrete actions."""

    def __init__(self, state_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.log_softmax(self.net(x), dim=-1)  # Log-probabilities

    def get_entropy(self, x: torch.Tensor) -> torch.Tensor:
        """Policy entropy H(pi). Higher = more exploratory."""
        log_probs = self.forward(x)
        probs = log_probs.exp()
        return -(probs * log_probs).sum(dim=-1).mean()


class ValueNetwork(nn.Module):
    """Baseline value network: outputs V(s)."""

    def __init__(self, state_dim: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def run_reinforce(
    n_episodes: int = 400,
    gamma: float = 0.99,
    policy_lr: float = 1e-3,
    value_lr: float = 5e-3,
    use_baseline: bool = True,
    normalize_returns: bool = True,
    entropy_coeff: float = 0.01,
) -> tuple:
    """REINFORCE with optional value baseline and entropy regularization."""
    policy = PolicyNetwork(4, 2).to(device)
    policy_opt = optim.Adam(policy.parameters(), lr=policy_lr)

    value_net = ValueNetwork(4).to(device) if use_baseline else None
    value_opt = optim.Adam(value_net.parameters(), lr=value_lr) if use_baseline else None

    episode_rewards = []
    entropy_log = []

    for ep in range(n_episodes):
        # Collect one full episode
        states, actions, rewards = [], [], []
        state = cartpole_reset()
        done = False

        while not done:
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                log_probs = policy(s_t)
                probs = log_probs.exp().squeeze().cpu().numpy()
            action = np.random.choice(2, p=probs)
            next_state, reward, done = cartpole_step(state, action)
            states.append(state.copy())
            actions.append(action)
            rewards.append(reward)
            state = next_state

        # Compute discounted returns G_t = sum_{k>=t} gamma^{k-t} r_{k+1}
        T = len(rewards)
        returns = np.zeros(T)
        G = 0.0
        for t in reversed(range(T)):
            G = rewards[t] + gamma * G
            returns[t] = G

        # Normalize returns to reduce variance
        if normalize_returns and T > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        states_t = torch.FloatTensor(np.array(states)).to(device)
        actions_t = torch.LongTensor(actions).to(device)
        returns_t = torch.FloatTensor(returns).to(device)

        # Compute baseline advantage
        if use_baseline and value_net is not None:
            baseline = value_net(states_t).detach()
            advantages = returns_t - baseline
        else:
            advantages = returns_t

        # Policy gradient update
        log_probs_t = policy(states_t)
        selected_log_probs = log_probs_t.gather(1, actions_t.unsqueeze(1)).squeeze(1)
        policy_loss = -(selected_log_probs * advantages.detach()).mean()

        # Entropy bonus to prevent premature convergence
        entropy = -(log_probs_t.exp() * log_probs_t).sum(dim=-1).mean()
        total_loss = policy_loss - entropy_coeff * entropy

        policy_opt.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        policy_opt.step()

        # Value network update (minimize MSE to discounted returns)
        if use_baseline and value_net is not None and value_opt is not None:
            value_pred = value_net(states_t)
            value_loss = F.mse_loss(value_pred, returns_t.detach())
            value_opt.zero_grad()
            value_loss.backward()
            value_opt.step()

        episode_rewards.append(sum(rewards))
        entropy_log.append(entropy.item())

        if (ep + 1) % 100 == 0:
            print(f'Ep {ep+1}: reward={np.mean(episode_rewards[-20:]):.1f}, '
                  f'entropy={np.mean(entropy_log[-20:]):.3f}')

    return episode_rewards, entropy_log


torch.manual_seed(42); np.random.seed(42)
reinforce_rewards, entropy_curve = run_reinforce(n_episodes=400)
print(f'\nREINFORCE+baseline final performance (last 50): {np.mean(reinforce_rewards[-50:]):.1f}')

## Real-World Example 1: Continuous Action Space — Gaussian Policy

Output mu(s) and log_sigma(s) from the policy network.  
Sample action ~ N(mu, sigma^2). Train on 1D continuous control task.

In [ ]:
class ContinuousControl1D:
    """1D continuous control: state=position in [-1,1], action=force in [-1,1]. Goal: reach 0.8."""

    def __init__(self):
        self.goal = 0.8
        self.reset()

    def reset(self) -> np.ndarray:
        self.pos = float(np.random.uniform(-1.0, -0.5))
        self.vel = 0.0
        return np.array([self.pos, self.vel])

    def step(self, action: float):
        action = np.clip(action, -1.0, 1.0)
        self.vel = 0.9 * self.vel + 0.1 * action  # Damped integration
        self.pos = np.clip(self.pos + self.vel, -1.0, 1.0)
        dist = abs(self.pos - self.goal)
        done = dist < 0.05
        reward = 1.0 if done else -dist
        return np.array([self.pos, self.vel]), reward, done


class GaussianPolicyNetwork(nn.Module):
    """Gaussian policy: outputs mu and log_sigma for continuous action."""

    def __init__(self, state_dim: int, hidden: int = 32):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(state_dim, hidden), nn.Tanh())
        self.mu_head = nn.Linear(hidden, 1)       # Mean of Gaussian
        self.log_sigma = nn.Parameter(torch.zeros(1))  # Learnable log std

    def forward(self, x: torch.Tensor) -> tuple:
        h = self.shared(x)
        mu = torch.tanh(self.mu_head(h))  # Bound to [-1, 1]
        sigma = self.log_sigma.exp().clamp(0.01, 1.0)  # Positive std
        return mu, sigma

    def log_prob(self, x: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        """Compute log pi(action|state) under Gaussian policy."""
        mu, sigma = self.forward(x)
        dist = torch.distributions.Normal(mu, sigma)
        return dist.log_prob(action).squeeze()

    def sample_action(self, x: torch.Tensor) -> tuple:
        """Sample action and return (action, log_prob)."""
        mu, sigma = self.forward(x)
        dist = torch.distributions.Normal(mu, sigma)
        action = dist.sample()
        log_prob = dist.log_prob(action).squeeze()
        return action.squeeze().item(), log_prob


def run_continuous_reinforce(n_episodes: int = 400, gamma: float = 0.99, lr: float = 5e-3) -> list:
    """REINFORCE with Gaussian policy on continuous 1D control."""
    env = ContinuousControl1D()
    policy = GaussianPolicyNetwork(state_dim=2).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    rewards_hist = []

    for ep in range(n_episodes):
        state = env.reset()
        log_probs_ep, rewards_ep = [], []
        done = False; steps = 0

        while not done and steps < 100:
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            action, log_prob = policy.sample_action(s_t)
            next_state, reward, done = env.step(action)
            log_probs_ep.append(log_prob)
            rewards_ep.append(reward)
            state = next_state
            steps += 1

        # Compute returns
        T = len(rewards_ep)
        returns = np.zeros(T)
        G = 0.0
        for t in reversed(range(T)):
            G = rewards_ep[t] + gamma * G
            returns[t] = G

        # Normalize returns
        if T > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        returns_t = torch.FloatTensor(returns).to(device)
        log_probs_t = torch.stack(log_probs_ep)

        # Policy gradient loss for Gaussian policy
        loss = -(log_probs_t * returns_t).mean()
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()

        rewards_hist.append(sum(rewards_ep))

    return rewards_hist


torch.manual_seed(42); np.random.seed(42)
continuous_rewards = run_continuous_reinforce(n_episodes=400)
print(f'Continuous REINFORCE final performance (last 50): {np.mean(continuous_rewards[-50:]):.3f}')

## Real-World Example 2: Variance Analysis — Baseline vs No Baseline vs Normalized

Run REINFORCE with three configurations and compare the variance of the policy gradient estimates.

In [ ]:
configs = [
    {'use_baseline': False, 'normalize_returns': False, 'name': 'No baseline, no norm'},
    {'use_baseline': False, 'normalize_returns': True,  'name': 'No baseline, normalized'},
    {'use_baseline': True,  'normalize_returns': True,  'name': 'Baseline + normalized'},
]

variance_results = {}
n_seeds = 5

for cfg in configs:
    all_rewards = []
    for seed in range(n_seeds):
        torch.manual_seed(seed); np.random.seed(seed)
        r, _ = run_reinforce(
            n_episodes=200,
            use_baseline=cfg['use_baseline'],
            normalize_returns=cfg['normalize_returns'],
        )
        all_rewards.append(r)
    variance_results[cfg['name']] = np.array(all_rewards)

print('Variance Comparison (last 50 episodes, std across 5 seeds):')
print(f'{"Configuration":<35} {"Mean Reward":<15} {"Std Reward":<15}')
print('-' * 65)
for name, arr in variance_results.items():
    mean_per_seed = arr[:, -50:].mean(axis=1)
    print(f'{name:<35} {mean_per_seed.mean():<15.2f} {mean_per_seed.std():<15.2f}')

## Real-World Example 3: Actor-Critic (Simple) — Replace G_t with TD Estimate

Replace Monte Carlo G_t with the TD advantage r + gamma*V(s') - V(s).  
This is the simplest Actor-Critic: per-step updates, much lower variance.

In [ ]:
class SimpleActorCritic(nn.Module):
    """Shared-base actor-critic network."""

    def __init__(self, state_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        # Shared feature extractor
        self.base = nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU())
        # Policy head: log-probabilities
        self.policy_head = nn.Linear(hidden, n_actions)
        # Value head: scalar V(s)
        self.value_head = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> tuple:
        h = self.base(x)
        log_probs = F.log_softmax(self.policy_head(h), dim=-1)
        value = self.value_head(h).squeeze(-1)
        return log_probs, value


def run_simple_actor_critic(
    n_episodes: int = 400,
    gamma: float = 0.99,
    lr: float = 1e-3,
    entropy_coeff: float = 0.01,
    value_coeff: float = 0.5,
) -> list:
    """One-step actor-critic on CartPole: per-step TD updates."""
    ac = SimpleActorCritic(4, 2).to(device)
    optimizer = optim.Adam(ac.parameters(), lr=lr)
    rewards_hist = []

    for ep in range(n_episodes):
        state = cartpole_reset()
        total_reward = 0.0
        done = False
        steps = 0

        while not done and steps < 500:
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            log_probs, value = ac(s_t)

            # Sample action from policy
            probs = log_probs.exp().squeeze()
            action = torch.multinomial(probs, 1).item()

            next_state, reward, done = cartpole_step(state, action)
            total_reward += reward

            # TD advantage: r + gamma * V(s') - V(s)
            ns_t = torch.FloatTensor(next_state).unsqueeze(0).to(device)
            with torch.no_grad():
                _, next_value = ac(ns_t)
                td_target = reward + gamma * next_value * (1.0 - float(done))

            advantage = (td_target - value).detach()

            # Actor loss: -log pi(a|s) * advantage
            selected_log_prob = log_probs[0, action]
            actor_loss = -selected_log_prob * advantage

            # Critic loss: MSE between V(s) and TD target
            critic_loss = F.mse_loss(value, td_target.detach())

            # Entropy bonus
            entropy = -(log_probs.exp() * log_probs).sum()

            # Combined loss
            loss = actor_loss + value_coeff * critic_loss - entropy_coeff * entropy

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ac.parameters(), max_norm=1.0)
            optimizer.step()

            state = next_state
            steps += 1

        rewards_hist.append(total_reward)

    return rewards_hist


torch.manual_seed(42); np.random.seed(42)
ac_rewards = run_simple_actor_critic(n_episodes=400)
print(f'Actor-Critic final performance (last 50): {np.mean(ac_rewards[-50:]):.1f}')

## Comparison: REINFORCE vs A2C Sample Efficiency and Variance

Plot learning curves for REINFORCE (no baseline), REINFORCE+baseline, and Actor-Critic.

In [ ]:
def smooth(arr, w=20):
    return np.convolve(arr, np.ones(w) / w, mode='valid')


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: CartPole Learning Curves ---
ax = axes[0]
ax.plot(smooth(reinforce_rewards), label='REINFORCE+baseline', color='blue')
ax.plot(smooth(ac_rewards), label='Actor-Critic (TD)', color='green')
ax.set_title('CartPole: Policy Gradient Methods')
ax.set_xlabel('Episode'); ax.set_ylabel('Episode Reward (smoothed)')
ax.legend()

# --- Plot 2: Policy Entropy ---
ax = axes[1]
ax.plot(smooth(entropy_curve, 10), color='purple', label='REINFORCE entropy')
ax.axhline(np.log(2), color='gray', linestyle='--', label='Max entropy (uniform)')
ax.set_title('Policy Entropy Over Training')
ax.set_xlabel('Episode'); ax.set_ylabel('Entropy (nats)')
ax.legend()

# --- Plot 3: Variance comparison ---
ax = axes[2]
names = list(variance_results.keys())
means = [variance_results[n][:, -50:].mean(axis=1).mean() for n in names]
stds = [variance_results[n][:, -50:].mean(axis=1).std() for n in names]
x_pos = np.arange(len(names))
ax.bar(x_pos, means, yerr=stds, capsize=5, color=['red', 'orange', 'green'], alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(['No base\nNo norm', 'No base\nNorm', 'Base+\nNorm'], fontsize=9)
ax.set_title('Variance Reduction (mean +/- std, 5 seeds)')
ax.set_ylabel('Mean Reward (last 50 eps)')

plt.suptitle('Policy Gradient Methods Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/policy_gradient_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/policy_gradient_comparison.png')

print('\n=== Summary ===')
print(f'Bandit REINFORCE optimal arm %: {np.mean(reinf_opt[-200:])*100:.1f}%')
print(f'Bandit Epsilon-greedy optimal %: {np.mean(eg_opt[-200:])*100:.1f}%')
print(f'CartPole REINFORCE+baseline: {np.mean(reinforce_rewards[-50:]):.1f}')
print(f'CartPole Actor-Critic (TD):  {np.mean(ac_rewards[-50:]):.1f}')

## Key Takeaways

**Core idea:** REINFORCE directly optimizes the policy by gradient ascent: increase the probability of actions that led to above-average returns. The log-gradient trick ∇log π(a|s) enables unbiased gradient estimation without differentiating through the environment.

**Variants and when to use:**

| Method | Return Estimate | Variance | Update Frequency | Use when |
|--------|---------------|----------|-----------------|----------|
| REINFORCE | Monte Carlo G_t | Very High | Per episode | Debugging, simple tasks |
| REINFORCE + baseline | G_t - V(s) | High | Per episode | Discrete action, episodic |
| Actor-Critic (TD) | r + gamma*V(s') | Medium | Per step | Any task, sample efficient |
| Gaussian policy | N(mu, sigma^2) | Depends on base | Per episode/step | Continuous action |

**Common failure modes:**
- No baseline: gradient estimates too noisy, reward curves look random — always add baseline
- Policy entropy collapse: policy becomes deterministic too early — add entropy coefficient 0.01
- Return scale mismatch: G_t in [0, 1000] causes gradient overflow — normalize returns per batch

**Related concepts:**
- [10-actor-critic](./10-actor-critic.ipynb) — adds GAE and shared networks for production use
- [08-deep-q-networks](./08-deep-q-networks.ipynb) — value-based alternative for discrete actions
- [06-q-learning](./06-q-learning.ipynb) — tabular analogue; same goal, different parameterization

## Exercises

1. **Entropy sweep:** Train REINFORCE+baseline with entropy_coeff in {0, 0.001, 0.01, 0.1}. Plot policy entropy curves and final reward. What is the minimum beta that prevents entropy collapse?
2. **Continuous action analysis:** Modify GaussianPolicyNetwork to output both mu and log_sigma as separate network heads (instead of shared log_sigma parameter). Does per-state variance improve performance?
3. **Credit assignment:** Log which timestep actions receive the highest gradient magnitude in a 200-step CartPole episode. Is it uniform, or do early actions get higher gradients? How does this relate to the importance of eligibility traces?
4. **Variance quantification:** After each episode, compute the variance of G_t across timesteps for REINFORCE vs G_t - V(s). Plot variance per episode. How many episodes until baseline reduces variance by 50%?